# GENERATE NEW QUESTIONS FROM DB FILE

In [103]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
%matplotlib inline

from datetime import date
today = str(date.today())

In [104]:
_files = os.listdir('../data/bkup')

_files[-1]

'2022-06-10_eventsDB.csv'

In [105]:
master_file = f'../data/bkup/{_files[-1]}'
df = pd.read_csv(master_file, encoding = "ISO-8859-1")
df = df.replace(r'^\s+$', np.nan, regex=True) #get rid of unwanted blank spaces in cells...

In [106]:
print(df.shape, df.columns)

(299, 35) Index(['eventID', 'stem', 'event', 'details', 'wikipedia', 'mnemonic',
       'itemDifficulty', 'categories', 'America', 'Europe', 'Asia', 'Africa',
       'China', 'Discovery', 'Britain', 'France', 'Greek', 'India',
       'Invention', 'MiddleEast', 'prehistory', 'Religion', 'Roman', 'Royalty',
       'Russia', 'Science', 'SouthAmerica', 'Wars', 'World', 'YearNum',
       'diffCat', 'timePeriod', 'altsMid', 'altsHard', 'altsEasy'],
      dtype='object')


# Logic Used to Generate Questions

1. Pick a question at random.
2. search for other questions at fixed distances of [0, 50, 100, 200, 300, 400, 500, 1000] and * -1
3. If the two questions don't have the same set of categories, print them both out.

In [107]:
steps = [50, 100, 200, 300, 400, 500, 1000]

steps += [x*(-1) for x in steps]

steps = set(steps)
print(steps)

{100, 200, 1000, 300, -500, -50, 400, -400, 50, 500, -300, -200, -1000, -100}


In [172]:
categs =['America', 'Europe', 'Asia', 'Africa',
       'China', 'Discovery', 'Britain', 'France', 'Greek', 'India',
       'Invention', 'MiddleEast', 'prehistory', 'Religion', 'Roman', 'Royalty',
       'Russia', 'Science', 'SouthAmerica', 'Wars', 'World',]

possessive = {"France": "French", "Russia": "Russian", 
              "America":"American", "India": "Indian", 
              "Europe":"European", "China":"Chinese",
             "Science": "Scientific",
             }

categ_set1 = ['Wars', 'Inventions']

possessive_categories = possessive.keys()

In [174]:
import random

def create_new_question():
    
    qsets = []
    
    yns = df['YearNum'].values
    idx = random.randint(0, len(yns)-1) 
    main_q = df.loc[idx]

    for step in steps:
        another_year=main_q['YearNum'] + step                
        if another_year in yns:
            indices = list(np.where(df['YearNum']==another_year)[0])
            for _aidx in indices:
                another_q = df.loc[_aidx]
                for c in categs: # Find a common category
                    if (main_q[c]==1) and (another_q[c]==1):
                        #found one
                        if pair_is_not_already_present(qsets, another_q['eventID']):
                            pair = {}
                            pair['ID1'] = main_q['eventID']
                            pair['ID2'] = another_q['eventID']
                            pair['Category'] = c
                            pair['First'] = main_q['event']
                            pair['Second'] = another_q['event']
                            pair["Year1"] = main_q['YearNum']
                            pair["Year2"] = another_q['YearNum']
                            pair['Diff'] = step
                            qsets.append(pair)

    return qsets


def pair_is_not_already_present(qsets, id2):
    
    for qset in qsets:
        if qset['ID2'] == id2:
            return 0
        
    return 1

def verbalize(pair):
    done = False    
    categ = pair['Category']
    _diff = pair['Diff']
    ev1 = pair['First']
    ev2 = pair['Second']
    yr1 = pair['Year1']
    yr2 = pair['Year2']
    

    print(" ")
    if categ in categ_set1:
        if _diff>0:
            print(f"Two {categ} separated by {_diff} years -- {ev1} and -- {ev2} in {yr2}")    
        if _diff<=0:
            print(f"Two {categ} separated by {-1*_diff} years -- {ev2} and -- {ev1} in {yr1}")    
        done = True
    if categ in possessive_categories:
        if _diff>0:
            print(f"Two {possessive[categ]} events separated by {_diff} years -- {ev1} and -- {ev2} in {yr2}")
        if _diff<=0:
            print(f"Two {possessive[categ]} events separated by {-1*_diff} years -- {ev2} and -- {ev1} in {yr1}")
        done = True

        
    if not done: #Catch-all
        print(f"{categ}:")          
        if _diff>0:
            print(f"Exactly {pair['Diff']} years after")
        else:
            print(f"Exactly {-1*pair['Diff']} years before")        
        print(f"{pair['First']}")
        print(f"{pair['Second']}")    
        
        
    print(" ")
        

In [176]:
qsets = create_new_question()

for qset in qsets:
    #print(qset)
    verbalize(qset)


 
Two European events separated by 400 years -- Turks captured Constantinople; Renaissance in Europe. and -- Crimean War -- Russia fights the Ottoman Empire, Britain and France. in 1853
 
 
Asia:
Exactly 500 years after
Turks captured Constantinople; Renaissance in Europe.
Death of Stalin; Mt. Everest conquered for the first time.
 
